## Baseline scores on semart using CLIP

In [113]:
%load_ext autoreload
%autoreload 2

import torch.nn.functional as F

import open_clip 
import numpy as np
import pandas as pd


from src.model import SheafMultimodalGNN
from src.utils import *
from src.data import *
# from src.ClusterData import ClusterData, ClusterLoader
from torch_geometric.data import DataLoader
from src.metrics import *

#triplets = '../artistic_sheaf/data/testing_elements.json'
triplets = 'data/triplets_semart_test_csv.json'
loaded_data = load_json_data(triplets)#[:5000]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded 9914 triplets from data/triplets_semart_test_csv.json


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
print(f"Using device: {device}")
seed_everything(seed=42)

# Load tokenizer and preprocessing
tokenizer = open_clip.get_tokenizer('ViT-B-32')
_, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', 
                                                         pretrained='laion2b_s34b_b79k')
# Initialize the model
model = SheafMultimodalGNN(
    latent_dim=512,
    edge_attr_dim=512,
    num_layers=3,
    step_size=1.0,
    lr=1e-4,
    test=False,
    clip_grad=True,
    device='cuda' if torch.cuda.is_available() else 'mps'
)
    
# Load checkpoint
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=43-val_loss=5.36.ckpt", map_location=device) # sheafCLIP frozen CLIP
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=23-val_loss=5.41.ckpt", map_location=device) # no sheaf frozen CLIP
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=01-val_loss=3.50.ckpt", map_location=device) # sheafCLIP CLIP grad
checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=10-val_loss=9.17.ckpt", map_location=device) # new version
	
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)
model.eval()
print()

Using device: cuda



### Testing on normal dataset

In [5]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data, preprocess, tokenizer, base_folder='../', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

100%|██████████| 9914/9914 [06:18<00:00, 26.20it/s]


9914


In [37]:
list(test_node_to_id.items())[:10]

[('Images/41294-10ladisl.jpg', 0),
 ("Of the Hungarian kings St Ladislas is perhaps the one most often represented in post-medieval frescoes, altar paintings and statues. In the Middle Ages he was associated with the ideal of chivalry and the legends which gathered around him determined the manner of his representation. Thus the knightly armour and the battle-axe have become permanent attributes, in addition to the crown and the orb.The painting illustrated here, dating from the late sixteenth century, is only superficially linked with medieval portraits of St. Ladislas. The king is shown seated on a throne wearing an ample, richly embroidered cloak studded with pearls round the hem. In the foreground is a voluted and foiled shield with the national emblem, stylized in harmony with the throne. The background is designed to suggest the interior of a Renaissance palace. Through opening on either side of the wall behind the throne - draped with embroidered hangings - there is a view of la

In [38]:
#clip_texts = get_clip_texts(loaded_data, 'item2', get_tokenizer('ViT-B-32'), model)
clip_images = []
clip_texts = []
for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, split='sheaf', check_images_=True)
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings = model(x_img, x_text, edge_index, edge_attr)
        print(embeddings.shape)
        clip_images.append(F.normalize(embeddings[: len(edge_attr), :], dim=1))
        clip_texts.append(F.normalize(embeddings[len(edge_attr):, :], dim=1))
        
clip_images = torch.cat(clip_images, dim=0)
clip_texts = torch.cat(clip_texts, dim=0)

print(f"Extracted {len(clip_texts)} text embeddings, each of shape {clip_texts[0].shape}")
print(f"Extracted {len(clip_images)} image embeddings, each of shape {clip_images[0].shape}")

clip_images = clip_images.cpu().detach().numpy()
clip_texts = clip_texts.cpu().detach().numpy()

torch.Size([1069, 3, 224, 224]) torch.Size([5153, 77]) torch.Size([2, 9914]) torch.Size([9914, 77])
Checking maps tensor([[-0.0860],
        [-0.0726],
        [-0.0726],
        [-0.0726],
        [-0.0726]], device='cuda:0')
Checking maps tensor([[-0.0335],
        [-0.0303],
        [-0.0303],
        [-0.0303],
        [-0.0303]], device='cuda:0')
Checking maps tensor([[0.3129],
        [0.1481],
        [0.1481],
        [0.1481],
        [0.1481]], device='cuda:0')
out shape torch.Size([9914, 512]) torch.Size([19828, 1])
Embeddings before change: tensor([[-0.4872,  0.5849,  1.2769, -0.4235, -0.0690],
        [-0.4872,  0.5849,  1.2769, -0.4235, -0.0690],
        [-0.4872,  0.5849,  1.2769, -0.4235, -0.0690],
        [-0.4872,  0.5849,  1.2769, -0.4235, -0.0690],
        [-0.4872,  0.5849,  1.2769, -0.4235, -0.0690]], device='cuda:0')
torch.Size([19828, 512])
Extracted 9914 text embeddings, each of shape torch.Size([512])
Extracted 9914 image embeddings, each of shape torch.Size([

In [111]:
clip_images[:, :10]

array([[ 0.02582587, -0.01396446, -0.0663061 , ..., -0.0404249 ,
         0.01090377,  0.08577424],
       [-0.01518888, -0.02580048,  0.02433934, ...,  0.00816267,
        -0.00441959,  0.01311707],
       [-0.01518888, -0.02580048,  0.02433934, ...,  0.00816267,
        -0.00441959,  0.01311707],
       ...,
       [-0.01016405, -0.11504049,  0.03616071, ...,  0.00373962,
        -0.01132053,  0.02833858],
       [ 0.02481285, -0.0434707 ,  0.0711196 , ..., -0.02321719,
         0.00976822, -0.02963652],
       [-0.07690574, -0.01687549,  0.03203066, ...,  0.04240893,
         0.00320193, -0.03442301]], dtype=float32)

In [112]:
clip_texts[:, :10]

array([[-0.00676709,  0.00820725, -0.03396695, ...,  0.04225329,
        -0.00202131, -0.0194915 ],
       [-0.0198712 , -0.05863941, -0.01077465, ...,  0.00949008,
         0.02259532, -0.06752693],
       [ 0.00353377,  0.01611619,  0.05416768, ..., -0.07377476,
        -0.01263947, -0.00808294],
       ...,
       [-0.01133225, -0.1144921 ,  0.03707119, ...,  0.00513697,
        -0.01144091,  0.02700376],
       [ 0.04901365, -0.00150688,  0.02740304, ..., -0.02137951,
         0.01700652, -0.07634311],
       [-0.02220895,  0.0562055 , -0.0291392 , ...,  0.04460098,
        -0.00285841,  0.09158838]], dtype=float32)

In [ ]:
# # save embeddings images and test

# np.save('data/clip_images_grad_clip.npy', clip_images)
# np.save('data/clip_texts_grad_clip.npy', clip_texts)


In [ ]:
# clip_images = np.load('data/clip_images_grad_clip.npy')
# clip_texts = np.load('data/clip_texts_grad_clip.npy')

In [9]:
# take a subset of the image embeddings and plot them with plotly interactively (in 2D using umap) showing the edge_index[0, i] on hover
import umap
import plotly.express as px
reducer = umap.UMAP()
#from sklearn.decomposition import PCA
#reducer = PCA(n_components=2)

embedding_2d = reducer.fit_transform(np.concatenate([clip_images, clip_texts], axis=0) ) # take only first 
fig = px.scatter(x=embedding_2d[:, 0], y=embedding_2d[:, 1],
                hover_data=[np.concatenate([np.arange(len(clip_images)), np.arange(len(clip_texts))], axis=0),
                            np.concatenate([test_graph_data.edge_index[0, :len(clip_images)].cpu().numpy(), test_graph_data.edge_index[1, :len(clip_texts)].cpu().numpy()], axis=0)], 
                color=['images']*(len(clip_images)) + ['text']*(len(clip_texts)))
fig.show()

### Image-to-text retrieval	and Text-to-image retrieval		
r@1	r@5	r@10	

In [128]:
adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data) 
print(f"Adjacency matrix shape: {adj_matrix.shape}")

Adjacency matrix shape: (7865, 5337)


In [129]:
sim_matrix = get_sim_matrix([t["item1"] + t["link"] for t in loaded_data], 
                            [t["item2"] + t["link"] for t in loaded_data], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
print(f"Similarity matrix shape: {sim_matrix.shape}")

Similarity matrix shape: (7865, 5337)


In [130]:
compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

{'t2i_precision@1': tensor(0.0245),
 't2i_recall@1': tensor(0.0198),
 't2i_ndcg@1': tensor(0.0245),
 't2i_precision@5': tensor(0.0184),
 't2i_recall@5': tensor(0.0672),
 't2i_ndcg@5': tensor(0.0473),
 't2i_precision@10': tensor(0.0170),
 't2i_recall@10': tensor(0.1218),
 't2i_ndcg@10': tensor(0.0649),
 'i2t_precision@1': tensor(0.2234),
 'i2t_recall@1': tensor(0.2213),
 'i2t_ndcg@1': tensor(0.2234),
 'i2t_precision@5': tensor(0.0822),
 'i2t_recall@5': tensor(0.4013),
 'i2t_ndcg@5': tensor(0.3187),
 'i2t_precision@10': tensor(0.0493),
 'i2t_recall@10': tensor(0.4765),
 'i2t_ndcg@10': tensor(0.3430),
 'mean_precision@1': tensor(0.1240),
 'mean_recall@1': tensor(0.1206),
 'mean_ndcg@1': tensor(0.1240),
 'mean_precision@5': tensor(0.0503),
 'mean_recall@5': tensor(0.2343),
 'mean_ndcg@5': tensor(0.1830),
 'mean_precision@10': tensor(0.0331),
 'mean_recall@10': tensor(0.2992),
 'mean_ndcg@10': tensor(0.2039)}

In [28]:
#print('uniformity images', uniformity(torch.tensor(clip_images)))
#print('uniformity texts', uniformity(torch.tensor(clip_texts)))
#print('alignment', alignment(torch.tensor(clip_images), torch.tensor(clip_texts)))

### Retrieval per type of relationship

In [18]:
#txt_to_idx

In [20]:
#img_to_idx

In [21]:
#adj_matrix

In [48]:
# numpy diagonal matrix with ones shape len(loaded_data) x len(loaded_data)
#adj_matrix = np.eye(len(loaded_data)).astype(int)
#print(adj_matrix.shape)
#adj_matrix

In [24]:
#sim_matrix

In [49]:
#sim_matrix = clip_images @ clip_texts.T
#sim_matrix

In [50]:
#compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

In [ ]:
#print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    #adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    #print(f"Adjacency matrix shape: {adj_matrix.shape}")
    #sim_matrix, clip_imgs_n, clip_txts_n = get_sim_matrix([t["item1"] for t in loaded_data_new], 
    #                        [t["item2"] for t in loaded_data_new], 
    #                        clip_images, clip_texts,
    #                        img_to_idx, txt_to_idx, out_emb=True)
    
    #idx12idx2 = {}
    #for i, img in enumerate(loaded_data_new):
    #    idx12idx2[i] = img_to_idx[img['item1']]
    #print('idx12idx2', idx12idx2)
    

In [131]:
from src.metrics import *

verbose = False
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    #if typ != 'timeframe':
    #    continue
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    indices_new = [i for i, l in enumerate(loaded_data) if l['link'] == typ]
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new) 
    print(adj_matrix.shape, 'shape adj matrix', len(set([l['item2'] for l in loaded_data_new])), 'unique text values')
    #print(indices_new[:5], loaded_data_new[:5])
    # subset of clip_img and clip_texts for loaded_data_new with typ = typ withou img_to_idx and txt_to_idx
    clip_imgs_n = clip_images[indices_new]
    clip_txts_n = clip_texts[indices_new]
    #print(clip_imgs_n[:5, :10])
    #print(list(img_to_idx.items())[:5])
    #sim_matrix = clip_imgs_n @ clip_txts_n.T
    sim_matrix = get_sim_matrix([t["item1"] + t["link"] for t in loaded_data_new], 
                                [t["item2"] + t["link"] for t in loaded_data_new], 
                                clip_imgs_n, clip_txts_n,
                                img_to_idx, txt_to_idx)
    print(sim_matrix.shape)
    #print(f"Similarity matrix shape: {sim_matrix.shape}")
    #print(compute_clip_metrics(torch.tensor(clip_imgs_n), torch.tensor(clip_txts_n), topk=[1, 5, 10]))
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))
    print('uniformity images', uniformity(torch.tensor(clip_imgs_n)))
    print('uniformity texts', uniformity(torch.tensor(clip_txts_n)))
    print('alignment', alignment(torch.tensor(clip_imgs_n), torch.tensor(clip_txts_n)))
    
    if verbose:
        # Print which query gives which recommendation (text or image path)
        # For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
        #indices_ladislas = [i for k,i in img_to_idx.items() if k == 'Images/41294-10ladisl.jpgcontent']
        #print(indices_ladislas)
        #_, js = np.where(adj_matrix[indices_ladislas, :] == 1)
        #print(js)
        #print(sim_matrix[indices_ladislas, js])
        #print(np.argmax(sim_matrix[indices_ladislas, :]))
        
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=min(5, len(loaded_data_new)))

        query_field = 'item1'  # image path
        rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}
        idx_to_img = {idx: img for img, idx in img_to_idx.items()}

        #idx_to_data_idxs = {}
        #for i, t in enumerate(loaded_data_new):
        #    if txt_to_idx[t["item2"] + t["link"]] in idx_to_data_idxs.keys():
        #        idx_to_data_idxs[txt_to_idx[t["item2"] + t["link"]]].append(i)
        #    else:
        #        idx_to_data_idxs[txt_to_idx[t["item2"] + t["link"]]] = [i]
        
        for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
            query = idx_to_img[i]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {[loaded_it[rec_field] for loaded_it in loaded_data_new if loaded_it['item1'] + loaded_it['link'] == query]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

Processing type: description
(963, 960) shape adj matrix 960 unique text values
(963, 960)
{'t2i_precision@1': tensor(0.0240), 't2i_recall@1': tensor(0.0240), 't2i_ndcg@1': tensor(0.0240), 't2i_precision@5': tensor(0.0179), 't2i_recall@5': tensor(0.0891), 't2i_ndcg@5': tensor(0.0570), 't2i_precision@10': tensor(0.0169), 't2i_recall@10': tensor(0.1667), 't2i_ndcg@10': tensor(0.0818), 'i2t_precision@1': tensor(0.0280), 'i2t_recall@1': tensor(0.0280), 'i2t_ndcg@1': tensor(0.0280), 'i2t_precision@5': tensor(0.0210), 'i2t_recall@5': tensor(0.1049), 'i2t_ndcg@5': tensor(0.0652), 'i2t_precision@10': tensor(0.0180), 'i2t_recall@10': tensor(0.1796), 'i2t_ndcg@10': tensor(0.0891), 'mean_precision@1': tensor(0.0260), 'mean_recall@1': tensor(0.0260), 'mean_ndcg@1': tensor(0.0260), 'mean_precision@5': tensor(0.0194), 'mean_recall@5': tensor(0.0970), 'mean_ndcg@5': tensor(0.0611), 'mean_precision@10': tensor(0.0174), 'mean_recall@10': tensor(0.1732), 'mean_ndcg@10': tensor(0.0855)}
uniformity images

## Evaluation with pseudo edge index

In [132]:
triplets = '../artistic_sheaf/data/full_triplets.json'
loaded_data = load_json_data(triplets)#[:9914]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

Loaded 19828 triplets from ../artistic_sheaf/data/full_triplets.json


In [133]:
# subselect elts in loaded_data where source == generated_i2t
loaded_data_i2t = [elt for elt in loaded_data if elt['source'] == 'generated_i2t']
print(f"Loaded {len(loaded_data_i2t)} triplets from {triplets} with source generated_i2t")
loaded_data_t2i = [elt for elt in loaded_data if elt['source'] == 'generated_t2i']
print(f"Loaded {len(loaded_data_t2i)} triplets from {triplets} with source generated_t2i")

Loaded 9914 triplets from ../artistic_sheaf/data/full_triplets.json with source generated_i2t
Loaded 9914 triplets from ../artistic_sheaf/data/full_triplets.json with source generated_t2i


In [134]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_i2t, preprocess, tokenizer, base_folder='../', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

100%|██████████| 9914/9914 [00:18<00:00, 534.64it/s] 


9914


In [135]:
clip_images_i2t = []
for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings = model(x_img, x_text, edge_index, edge_attr)
        clip_images_i2t.append(F.normalize(embeddings[: len(edge_attr), :], dim=1))
clip_images_i2t = torch.cat(clip_images_i2t, dim=0)

print(f"Extracted {len(clip_images_i2t)} image embeddings, each of shape {clip_images_i2t[0].shape}")

clip_images_i2t = clip_images_i2t.cpu().detach().numpy()

torch.Size([638, 3, 224, 224]) torch.Size([1054, 77]) torch.Size([2, 3304]) torch.Size([3304, 77])
Checking maps tensor([[-0.0114],
        [-0.0421],
        [-0.0575],
        [ 0.0158],
        [-0.0448]], device='cuda:0')
Checking maps tensor([[-0.0105],
        [-0.0660],
        [-0.0067],
        [-0.0019],
        [-0.0323]], device='cuda:0')
Checking maps tensor([[0.2452],
        [0.4452],
        [0.3488],
        [0.3037],
        [0.2575]], device='cuda:0')
out shape torch.Size([3304, 512]) torch.Size([6608, 1])
Embeddings before change: tensor([[-0.3491,  0.1836,  1.7039, -0.1788,  0.4289],
        [-0.2692,  1.4668,  1.1288,  0.5447, -0.2659],
        [ 0.4239,  1.8647,  0.4067, -2.5223,  0.1203],
        [-0.1368,  1.1940,  0.8963, -0.0275,  0.4629],
        [-0.3795,  1.2050,  0.4325,  0.3541, -0.0962]], device='cuda:0')
torch.Size([406, 3, 224, 224]) torch.Size([512, 77]) torch.Size([2, 3304]) torch.Size([3304, 77])
Checking maps tensor([[-0.0528],
        [-0.0453],


In [136]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_t2i, preprocess, tokenizer, base_folder='../', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

100%|██████████| 9914/9914 [00:22<00:00, 449.62it/s] 


9914


In [137]:
clip_texts_t2i = []

for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings = model(x_img, x_text, edge_index, edge_attr)
        clip_texts_t2i.append(F.normalize(embeddings[len(edge_attr):, :], dim=1))
clip_texts_t2i = torch.cat(clip_texts_t2i, dim=0)

print(f"Extracted {len(clip_texts_t2i)} text embeddings, each of shape {clip_texts_t2i[0].shape}")

clip_texts_t2i = clip_texts_t2i.cpu().detach().numpy()

torch.Size([765, 3, 224, 224]) torch.Size([3163, 77]) torch.Size([2, 3304]) torch.Size([3304, 77])
Checking maps tensor([[-0.0671],
        [-0.0686],
        [-0.0574],
        [ 0.0158],
        [-0.0448]], device='cuda:0')
Checking maps tensor([[-0.0022],
        [-0.0740],
        [-0.0114],
        [-0.0019],
        [-0.0323]], device='cuda:0')
Checking maps tensor([[0.3485],
        [0.4244],
        [0.3309],
        [0.3037],
        [0.2575]], device='cuda:0')
out shape torch.Size([3304, 512]) torch.Size([6608, 1])
Embeddings before change: tensor([[-1.5854e-01,  5.2255e-01,  9.1476e-01, -4.7399e-01,  1.3361e-01],
        [-1.0930e+00,  1.0559e-03,  1.4254e+00, -5.4986e-01,  6.2716e-02],
        [ 1.9534e-01,  3.7207e-01,  4.3889e-01, -1.0262e+00, -3.2580e-01],
        [-1.5971e-01,  1.1606e+00,  8.7646e-01, -5.0922e-02,  4.8684e-01],
        [-3.7644e-01,  1.1872e+00,  4.0615e-01,  3.4434e-01, -6.7802e-02]],
       device='cuda:0')
torch.Size([497, 3, 224, 224]) torch.Size([

In [138]:
from collections import defaultdict 
def reorder_predictions_by_link_item(
    predictions: np.ndarray,
    new_list,   # "data/full_triplets.json" (the list used to produce predictions)
    old_list,   # the new file with same links but different item2 assignments
    item = 'item2',  # which item to use for matching (default 'item2' for text predictions)
):
    """
    Reorder the text predictions to match the order of (link, item2) in the new triplet file.
    Assumes:
      - predictions_txt[i] corresponds to old_list[i]['item2'] with old_list[i]['link'].
      - Keys used for matching are (link, item2).
      - Handles duplicate (link, item2) by consuming old indices FIFO.
    """
    
    # Build mapping: (link, item2) -> queue of old indices
    pos_by_key = defaultdict(list)
    for idx, tr in enumerate(old_list):
        link = tr.get("link")
        item2 = tr.get(item)
        pos_by_key[(link, item2)].append(idx)

    print(len(pos_by_key), "unique (link,item) pairs in the old file.")
    # Build reorder indices to match new_list order
    
    reorder_indices = []
    missing = []
    for tr in new_list:
        key = (tr.get("link"), tr.get(item))
        if pos_by_key[key]:
            reorder_indices.append(pos_by_key[key].pop(0))  # consume one occurrence
        else:
            missing.append(key)

    if missing:
        # Raise for visibility; switch to a warning if partial overlap is expected.
        example = missing[:5]
        print(
            f"{len(missing)} (link,{item}) pairs in the new file were not found in the old predictions. "
            f"Examples: {example}"
        )

    # Reorder predictions
    idx_t = np.array(reorder_indices)
    print(reorder_indices[:10])
    predictions_reordered = predictions[idx_t]
    
    # print how many triplets (link, item1, item2) are the same in the old and new list by creating dictionaries
    # Build mapping: (link, item2) -> queue of old indices
    pos_by_key_all = defaultdict(list)
    for idx, tr in enumerate(old_list):
        link = tr.get("link")
        item2 = tr.get('item2')
        item1 = tr.get('item1')
        pos_by_key_all[(link, item2, item1)].append(idx)

    wrong = []
    for tr in new_list:
        key = (tr.get("link"), tr.get('item2'), tr.get('item1'))
        if pos_by_key_all[key]:
            reorder_indices.append(pos_by_key_all[key].pop(0))  # consume one occurrence
        else:
            wrong.append(key)

    print("Accuracy of preliminary matching (link, item1, item2):",
          1 - len(wrong) / len(new_list))

    return predictions_reordered


In [139]:
loaded_data = load_json_data("data/triplets_semart_test_csv.json")#[:9914]
print(f"Loaded {len(loaded_data)} triplets from data/triplets_semart_test_orig.json")

Loaded 9914 triplets from data/triplets_semart_test_orig.json


In [140]:
predictions_txt_new_order = reorder_predictions_by_link_item(
    clip_texts_t2i,
    new_list=loaded_data_t2i,  # use only the test portion of the loaded data
    old_list=loaded_data,
    item='item2'
)
print(predictions_txt_new_order.shape)
predictions_txt_new_order[:5]
    

5337 unique (link,item) pairs in the old file.
[0, 13, 27, 42, 74, 103, 112, 121, 130, 139]
Accuracy of preliminary matching (link, item1, item2): 0.08382085939076056
(9914, 512)


array([[-0.00676709,  0.00820725, -0.03396695, ...,  0.01606232,
         0.0491451 ,  0.01856392],
       [ 0.02463759,  0.02067691,  0.02040513, ..., -0.00913673,
         0.07535452, -0.02315984],
       [-0.04231564, -0.04348896,  0.04859691, ..., -0.02238578,
        -0.05060646,  0.00825453],
       [ 0.00445496,  0.025774  ,  0.00216472, ...,  0.01098963,
         0.05871863, -0.03612882],
       [ 0.0236448 ,  0.0134121 , -0.01985466, ...,  0.00667346,
         0.06877115, -0.03030667]], dtype=float32)

In [141]:
predictions_image_new_order = reorder_predictions_by_link_item(
    clip_images_i2t,
    new_list=loaded_data_i2t,  # use only the test portion of the loaded data
    old_list=loaded_data,
    item='item1',
)
print(predictions_image_new_order.shape)

7865 unique (link,item) pairs in the old file.
7397 (link,item1) pairs in the new file were not found in the old predictions. Examples: [('description', 'Images/39651-6eccehom.jpg'), ('description', 'Images/02938-st_luke.jpg'), ('description', 'Images/35872-pastoral.jpg'), ('description', 'Images/25363-bacchus.jpg'), ('description', 'Images/02938-st_luke.jpg')]
[2555, 3981, 7508, 42, 74, 6947, 112, 3264, 130, 6401]
Accuracy of preliminary matching (link, item1, item2): 0.1694573330643534
(2517, 512)


In [147]:
adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data) 
print(f"Adjacency matrix shape: {adj_matrix.shape}")
sim_matrix = get_sim_matrix([t["item1"] + t["link"] for t in loaded_data], 
                            [t["item2"] + t["link"] for t in loaded_data], 
                            clip_images_i2t, clip_texts_t2i,
                            img_to_idx, txt_to_idx)
print(f"Similarity matrix shape: {sim_matrix.shape}")
compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

Adjacency matrix shape: (7865, 5337)
Similarity matrix shape: (7865, 5337)


{'t2i_precision@1': tensor(0.0034),
 't2i_recall@1': tensor(0.0030),
 't2i_ndcg@1': tensor(0.0034),
 't2i_precision@5': tensor(0.0041),
 't2i_recall@5': tensor(0.0192),
 't2i_ndcg@5': tensor(0.0117),
 't2i_precision@10': tensor(0.0037),
 't2i_recall@10': tensor(0.0348),
 't2i_ndcg@10': tensor(0.0163),
 'i2t_precision@1': tensor(0.0047),
 'i2t_recall@1': tensor(0.0035),
 'i2t_ndcg@1': tensor(0.0047),
 'i2t_precision@5': tensor(0.0037),
 'i2t_recall@5': tensor(0.0144),
 'i2t_ndcg@5': tensor(0.0094),
 'i2t_precision@10': tensor(0.0039),
 'i2t_recall@10': tensor(0.0302),
 'i2t_ndcg@10': tensor(0.0148),
 'mean_precision@1': tensor(0.0040),
 'mean_recall@1': tensor(0.0033),
 'mean_ndcg@1': tensor(0.0040),
 'mean_precision@5': tensor(0.0039),
 'mean_recall@5': tensor(0.0168),
 'mean_ndcg@5': tensor(0.0106),
 'mean_precision@10': tensor(0.0038),
 'mean_recall@10': tensor(0.0325),
 'mean_ndcg@10': tensor(0.0156)}

In [146]:
verbose = True
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    print(f"Adjacency matrix shape: {adj_matrix.shape}")
    sim_matrix = get_sim_matrix([t["item1"] + t['link'] for t in loaded_data_new], 
                            [t["item2"] + t['link'] for t in loaded_data_new], 
                            clip_images_i2t, clip_texts_t2i,
                            img_to_idx, txt_to_idx)
    print(f"Similarity matrix shape: {sim_matrix.shape}")
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))

    if verbose:
        
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=min(5, len(loaded_data_new)))

        query_field = 'item1'  # image path
        rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}
        idx_to_img = {idx: img for img, idx in img_to_idx.items()}

        
        for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
            query = idx_to_img[i]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {[loaded_it[rec_field] for loaded_it in loaded_data_new if loaded_it['item1'] + loaded_it['link'] == query]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

Processing type: description
Loaded 963 triplets for type description
Adjacency matrix shape: (963, 960)
Similarity matrix shape: (963, 960)
{'t2i_precision@1': tensor(0.0135), 't2i_recall@1': tensor(0.0135), 't2i_ndcg@1': tensor(0.0135), 't2i_precision@5': tensor(0.0094), 't2i_recall@5': tensor(0.0469), 't2i_ndcg@5': tensor(0.0296), 't2i_precision@10': tensor(0.0091), 't2i_recall@10': tensor(0.0906), 't2i_ndcg@10': tensor(0.0432), 'i2t_precision@1': tensor(0.0083), 'i2t_recall@1': tensor(0.0083), 'i2t_ndcg@1': tensor(0.0083), 'i2t_precision@5': tensor(0.0091), 'i2t_recall@5': tensor(0.0457), 'i2t_ndcg@5': tensor(0.0260), 'i2t_precision@10': tensor(0.0098), 'i2t_recall@10': tensor(0.0976), 'i2t_ndcg@10': tensor(0.0426), 'mean_precision@1': tensor(0.0109), 'mean_recall@1': tensor(0.0109), 'mean_ndcg@1': tensor(0.0109), 'mean_precision@5': tensor(0.0093), 'mean_recall@5': tensor(0.0463), 'mean_ndcg@5': tensor(0.0278), 'mean_precision@10': tensor(0.0094), 'mean_recall@10': tensor(0.0941),